# 04b - Retrieval experiment: SEA-LION rerank

Experiment 1 of the SEA-LION comparison (see CLAUDE.md's Architecture decisions
-> Retrieval section). Keeps the committed BGE-M3 embeddings unchanged and swaps
only the rerank stage: score hybrid candidates by cosine similarity using
`aisingapore/SEA-LION-E5-Embedding-600M` -- purpose-built and SEA-BED-benchmarked
on Tamil/Burmese/Thai/Vietnamese/etc, unlike the two rerankers already compared
in `06_eval.ipynb` (`ms-marco-MiniLM-L-6-v2`, English-only; `bge-reranker-v2-m3`,
general multilingual).

Caveat: SEA-LION-E5-Embedding-600M is a bi-encoder (like BGE-M3), not a
cross-encoder like the other two rerankers -- it reranks by embedding cosine
similarity, not joint query-passage cross-attention. Not a strictly like-for-like
swap; kept in mind when reading the comparison below.

Reuses the same labeled query set and metrics as `06_eval.ipynb` so results are
directly comparable, and reruns dense/bm25/hybrid/hybrid_rerank/hybrid_rerank_bge
alongside the new `hybrid_rerank_sealion` method, so this notebook's saved output
doubles as a fresh baseline snapshot too.

## Step 1: Setup

Same setup as `06_eval.ipynb` (chunks, BGE-M3 embeddings, Chroma, BM25, the two
existing rerankers), plus the new SEA-LION embedding model and a bi-encoder
rerank function built from it.

In [1]:
import json
import re
from pathlib import Path

import chromadb
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

CHUNKS_PATH = Path("../data/processed/chunks.json")
EMBEDDINGS_PATH = Path("../data/processed/embeddings_bge_m3.npy")
CHUNK_IDS_PATH = Path("../data/processed/chunk_ids_bge_m3.json")
CHROMA_DIR = Path("../data/processed/chroma")

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RERANKER_MODEL_NAME_BGE = "BAAI/bge-reranker-v2-m3"
# Experiment 1: SEA-LION's SEA-BED-benchmarked embedding model, used as a
# bi-encoder reranker (cosine similarity, not cross-attention) -- purpose-built
# and evaluated on Tamil/Burmese/Thai/etc, unlike the two cross-encoders above.
SEALION_MODEL_NAME = "aisingapore/SEA-LION-E5-Embedding-600M"

chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
chunks_by_id = {chunk["chunk_id"]: chunk for chunk in chunks}

embeddings = np.load(EMBEDDINGS_PATH)
chunk_ids = json.loads(CHUNK_IDS_PATH.read_text(encoding="utf-8"))
chunk_id_to_idx = {cid: i for i, cid in enumerate(chunk_ids)}

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(name="bge_m3")

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
reranker = CrossEncoder(RERANKER_MODEL_NAME)
reranker_bge = CrossEncoder(RERANKER_MODEL_NAME_BGE)
sealion_model = SentenceTransformer(SEALION_MODEL_NAME)


def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


bm25 = BM25Okapi([tokenize(chunks_by_id[cid]["text"]) for cid in chunk_ids])


def dense_retrieve(query: str, top_k: int = 5) -> list[str]:
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=top_k)
    return list(results["ids"][0])


def bm25_retrieve(query: str, top_k: int = 5) -> list[str]:
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(zip(chunk_ids, scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [cid for cid, _ in ranked]


def hybrid_retrieve(query: str, top_k: int = 5) -> list[str]:
    dense_ids = dense_retrieve(query, top_k=10)
    bm25_ids = bm25_retrieve(query, top_k=10)
    scores: dict[str, float] = {}
    for ranked in (dense_ids, bm25_ids):
        for rank, cid in enumerate(ranked, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (60 + rank)
    return [cid for cid, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]]


def _cross_encoder_rerank(model: CrossEncoder, query: str, top_k: int) -> list[str]:
    candidates = hybrid_retrieve(query, top_k=10)
    pairs = [(query, chunks_by_id[cid]["text"]) for cid in candidates]
    scores = model.predict(pairs)
    reranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in reranked[:top_k]]


def hybrid_rerank_retrieve(query: str, top_k: int = 5) -> list[str]:
    return _cross_encoder_rerank(reranker, query, top_k)


def hybrid_rerank_bge_retrieve(query: str, top_k: int = 5) -> list[str]:
    return _cross_encoder_rerank(reranker_bge, query, top_k)


def hybrid_rerank_sealion_retrieve(query: str, top_k: int = 5) -> list[str]:
    candidates = hybrid_retrieve(query, top_k=10)
    # STS is the only named prompt documented on the SEA-LION-E5 model card,
    # used for both query and passage since this only needs a symmetric
    # cosine-similarity score for reranking, not asymmetric retrieval.
    query_embedding = sealion_model.encode(query, convert_to_numpy=True, prompt_name="STS")
    candidate_embeddings = sealion_model.encode(
        [chunks_by_id[cid]["text"] for cid in candidates],
        convert_to_numpy=True,
        prompt_name="STS",
    )
    similarities = candidate_embeddings @ query_embedding / (
        np.linalg.norm(candidate_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    reranked = sorted(zip(candidates, similarities), key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in reranked[:top_k]]


RETRIEVAL_METHODS = {
    "dense": dense_retrieve,
    "bm25": bm25_retrieve,
    "hybrid": hybrid_retrieve,
    "hybrid_rerank": hybrid_rerank_retrieve,
    "hybrid_rerank_bge": hybrid_rerank_bge_retrieve,
    "hybrid_rerank_sealion": hybrid_rerank_sealion_retrieve,
}

len(chunks), collection.count(), list(RETRIEVAL_METHODS)

c:\Users\rames\Documents\GitHub\migrantBuddy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3493.77it/s]
c:\Users\rames\Documents\GitHub\migrantBuddy\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rames\.cache\huggingface\hub\models--aisingapore--SEA-LION-E5-Embedding-600M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on

(16,
 16,
 ['dense',
  'bm25',
  'hybrid',
  'hybrid_rerank',
  'hybrid_rerank_bge',
  'hybrid_rerank_sealion'])

## Step 2: Labeled eval set

Identical `(query, correct_chunk_id)` pairs to `06_eval.ipynb` -- same ground
truth, so scores are comparable method-for-method across notebooks.

In [2]:
def find_chunk_id(document_slug: str, heading_contains: str) -> str:
    matches = [
        cid
        for cid in chunk_ids
        if document_slug in chunks_by_id[cid]["url"]
        and heading_contains.lower() in chunks_by_id[cid]["heading_path"].lower()
    ]
    assert len(matches) == 1, f"Expected exactly 1 match for {heading_contains!r}, got {matches}"
    return matches[0]


def find_chunk_id_by_document(document_slug: str) -> str:
    matches = [cid for cid in chunk_ids if document_slug in chunks_by_id[cid]["url"]]
    assert len(matches) == 1, f"Expected exactly 1 chunk for {document_slug!r}, got {matches}"
    return matches[0]


LABELED_QUERIES = [
    {
        "language": "en",
        "text": "How much overtime pay am I entitled to?",
        "correct_chunk_id": find_chunk_id("hours-of-work", "Overtime pay"),
    },
    {
        "language": "en",
        "text": "When must my employer pay my salary?",
        "correct_chunk_id": find_chunk_id("paying-salary", "How often salary must be paid"),
    },
    {
        "language": "ms",
        "text": "Bilakah majikan saya perlu bayar gaji saya?",
        "correct_chunk_id": find_chunk_id("paying-salary", "How often salary must be paid"),
    },
    {
        "language": "ta",
        "text": "எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?",
        "correct_chunk_id": find_chunk_id("hours-of-work", "Overtime pay"),
    },
    {
        "language": "my",
        "text": "ကျွန်တော် ဘယ်လောက် အချိန်ပိုခ ရထိုက်သလဲ",
        "correct_chunk_id": find_chunk_id("hours-of-work", "Overtime pay"),
    },
    {
        "language": "th",
        "text": "ฉันมีสิทธิ์ได้รับค่าล่วงเวลาเท่าไหร่",
        "correct_chunk_id": find_chunk_id("hours-of-work", "Overtime pay"),
    },
    {
        "language": "vi",
        "text": "Chủ sử dụng lao động của tôi phải trả lương khi nào?",
        "correct_chunk_id": find_chunk_id("paying-salary", "How often salary must be paid"),
    },
    {
        "language": "en",
        "text": "Who pays repatriation costs when my Work Permit ends?",
        "correct_chunk_id": find_chunk_id("work-permit-conditions", "employment ends"),
    },
    {
        "language": "en",
        "text": "How much medical insurance must my employer provide?",
        "correct_chunk_id": find_chunk_id("medical-insurance", "should cover"),
    },
    {
        "language": "en",
        "text": "How can I contact MOM?",
        "correct_chunk_id": find_chunk_id_by_document("contact-us"),
    },
]

for q in LABELED_QUERIES:
    print(f"[{q['language']}] {q['text']}\n  -> {q['correct_chunk_id']}\n")

[en] How much overtime pay am I entitled to?
  -> https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2

[en] When must my employer pay my salary?
  -> https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1

[ms] Bilakah majikan saya perlu bayar gaji saya?
  -> https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1

[ta] எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?
  -> https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2

[my] ကျွန်တော် ဘယ်လောက် အချိန်ပိုခ ရထိုက်သလဲ
  -> https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2

[th] ฉันมีสิทธิ์ได้รับค่าล่วงเวลาเท่าไหร่
  -> https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2

[vi] Chủ sử dụng lao động của tôi phải trả lương khi nào?
  -> https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1

[en] Who pays repatriation costs when my Work Permit ends?

## Step 3: Retrieval metrics

Same MRR / hit@k / precision@k / recall@k / nDCG@k harness as `06_eval.ipynb`,
now run across all six methods including `hybrid_rerank_sealion`.

In [3]:
import math
import time


def reciprocal_rank(ranked_ids: list[str], relevant_ids: set[str]) -> float:
    for i, cid in enumerate(ranked_ids, start=1):
        if cid in relevant_ids:
            return 1.0 / i
    return 0.0


def hit_at_k(ranked_ids: list[str], relevant_ids: set[str], k: int) -> float:
    return float(any(cid in relevant_ids for cid in ranked_ids[:k]))


def precision_at_k(ranked_ids: list[str], relevant_ids: set[str], k: int) -> float:
    top_k = ranked_ids[:k]
    return sum(1 for cid in top_k if cid in relevant_ids) / k


def recall_at_k(ranked_ids: list[str], relevant_ids: set[str], k: int) -> float:
    hits = sum(1 for cid in ranked_ids[:k] if cid in relevant_ids)
    return hits / len(relevant_ids)


def ndcg_at_k(ranked_ids: list[str], relevant_ids: set[str], k: int) -> float:
    dcg = sum(
        1.0 / math.log2(i + 1)
        for i, cid in enumerate(ranked_ids[:k], start=1)
        if cid in relevant_ids
    )
    ideal_hits = min(len(relevant_ids), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


METRIC_TOP_K = 5
EVAL_K = 3

per_query_results: dict[str, list[dict]] = {}

for method_name, retrieve_fn in RETRIEVAL_METHODS.items():
    rows = []
    for q in LABELED_QUERIES:
        relevant_ids = {q["correct_chunk_id"]}

        start = time.perf_counter()
        ranked_ids = retrieve_fn(q["text"], top_k=METRIC_TOP_K)
        latency_ms = (time.perf_counter() - start) * 1000

        rows.append(
            {
                "language": q["language"],
                "query": q["text"],
                "rr": reciprocal_rank(ranked_ids, relevant_ids),
                "hit@k": hit_at_k(ranked_ids, relevant_ids, EVAL_K),
                "precision@k": precision_at_k(ranked_ids, relevant_ids, EVAL_K),
                "recall@k": recall_at_k(ranked_ids, relevant_ids, EVAL_K),
                "ndcg@k": ndcg_at_k(ranked_ids, relevant_ids, EVAL_K),
                "latency_ms": latency_ms,
            }
        )
    per_query_results[method_name] = rows

results_table = []
for method_name, rows in per_query_results.items():
    n = len(rows)
    results_table.append(
        {
            "method": method_name,
            "MRR": sum(r["rr"] for r in rows) / n,
            f"hit@{EVAL_K}": sum(r["hit@k"] for r in rows) / n,
            f"precision@{EVAL_K}": sum(r["precision@k"] for r in rows) / n,
            f"recall@{EVAL_K}": sum(r["recall@k"] for r in rows) / n,
            f"nDCG@{EVAL_K}": sum(r["ndcg@k"] for r in rows) / n,
            "avg_latency_ms": sum(r["latency_ms"] for r in rows) / n,
        }
    )

for row in results_table:
    print(
        f"{row['method']:22s}  MRR={row['MRR']:.3f}  hit@{EVAL_K}={row[f'hit@{EVAL_K}']:.3f}  "
        f"precision@{EVAL_K}={row[f'precision@{EVAL_K}']:.3f}  recall@{EVAL_K}={row[f'recall@{EVAL_K}']:.3f}  "
        f"nDCG@{EVAL_K}={row[f'nDCG@{EVAL_K}']:.3f}  latency={row['avg_latency_ms']:.1f}ms"
    )

dense                   MRR=0.875  hit@3=0.900  precision@3=0.300  recall@3=0.900  nDCG@3=0.863  latency=136.9ms
bm25                    MRR=0.403  hit@3=0.600  precision@3=0.200  recall@3=0.600  nDCG@3=0.439  latency=0.1ms
hybrid                  MRR=0.783  hit@3=0.900  precision@3=0.300  recall@3=0.900  nDCG@3=0.813  latency=55.3ms
hybrid_rerank           MRR=0.425  hit@3=0.500  precision@3=0.167  recall@3=0.500  nDCG@3=0.426  latency=266.6ms
hybrid_rerank_bge       MRR=0.817  hit@3=1.000  precision@3=0.333  recall@3=1.000  nDCG@3=0.863  latency=4772.0ms
hybrid_rerank_sealion   MRR=0.950  hit@3=1.000  precision@3=0.333  recall@3=1.000  nDCG@3=0.963  latency=4871.9ms


## Reranker comparison & decision

| Method | MRR | hit@3 | nDCG@3 | Avg latency |
|---|---|---|---|---|
| dense (no rerank) | 0.875 | 0.900 | 0.863 | 137ms |
| bm25 | 0.403 | 0.600 | 0.439 | 0.1ms |
| hybrid (no rerank) | 0.783 | 0.900 | 0.813 | 55ms |
| hybrid_rerank (ms-marco) | 0.425 | 0.500 | 0.426 | 267ms |
| hybrid_rerank_bge | 0.817 | 1.000 | 0.863 | 4772ms |
| **hybrid_rerank_sealion** | **0.950** | **1.000** | **0.963** | 4872ms |

Per-query detail explains the gap: ms-marco fails outright (RR=0.000) on ms/ta/my/vi
-- expected, since it's English-only-trained and can't judge cross-lingual pairs.
bge-reranker-v2-m3 recovers most of that (perfect RR on ta/my/th) but still drops
rank on ms and vi (RR=0.333 each). SEA-LION is the only reranker to score a
perfect RR=1.000 on every non-English query, missing only on the one query all
three rerankers struggle with alike (medical-insurance, RR=0.500 across the
board -- likely a genuinely ambiguous query, not a language issue).

Latency-wise, bge-reranker-v2-m3 and SEA-LION cost almost the same (~4.7-4.9s per
query), so accuracy is the deciding factor here, not speed.

**Decision: `hybrid_rerank_sealion` is the chosen reranker.** It is the only
method that beats plain dense retrieval (0.875 MRR) on every metric, and clearly
outperforms the other two rerankers at no meaningful extra latency cost over
bge-reranker-v2-m3. Next step: apply the same swap in `05_generation.ipynb` once
ready (currently staged with bge-reranker-v2-m3 as an intermediate baseline).

## Step 4: Compare & save

Per-query breakdown (watch specifically for the Tamil/Burmese/Thai rows, where
SEA-LION's SEA-BED benchmark claims an edge), then save results to disk so this
run -- baseline methods plus the SEA-LION experiment -- can be compared against
experiment 2 (SEA-LION embeddings + rerank) later without needing to keep
notebooks open side by side.

In [4]:
print("Per-query breakdown (reciprocal rank, latency):\n")
for method_name, rows in per_query_results.items():
    print(f"--- {method_name} ---")
    for r in rows:
        print(f"  [{r['language']}] RR={r['rr']:.3f}  latency={r['latency_ms']:.1f}ms  {r['query'][:50]}")
    print()

best_method = max(results_table, key=lambda r: r["MRR"])
print(f"Best by MRR: {best_method['method']} (MRR={best_method['MRR']:.3f})")

Per-query breakdown (reciprocal rank, latency):

--- dense ---
  [en] RR=1.000  latency=863.6ms  How much overtime pay am I entitled to?
  [en] RR=1.000  latency=53.1ms  When must my employer pay my salary?
  [ms] RR=1.000  latency=55.3ms  Bilakah majikan saya perlu bayar gaji saya?
  [ta] RR=1.000  latency=58.3ms  எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?
  [my] RR=0.250  latency=60.0ms  ကျွန်တော် ဘယ်လောက် အချိန်ပိုခ ရထိုက်သလဲ
  [th] RR=1.000  latency=53.5ms  ฉันมีสิทธิ์ได้รับค่าล่วงเวลาเท่าไหร่
  [vi] RR=1.000  latency=60.3ms  Chủ sử dụng lao động của tôi phải trả lương khi nà
  [en] RR=1.000  latency=56.5ms  Who pays repatriation costs when my Work Permit en
  [en] RR=0.500  latency=56.0ms  How much medical insurance must my employer provid
  [en] RR=1.000  latency=52.8ms  How can I contact MOM?

--- bm25 ---
  [en] RR=0.333  latency=0.3ms  How much overtime pay am I entitled to?
  [en] RR=0.500  latency=0.1ms  When must my employer pay my salary?
  [ms] RR=0.500  latency=0.1ms 

In [5]:
import datetime

RESULTS_DIR = Path("../data/processed/eval_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

output = {
    "notebook": "04b_retrieval_sealion_rerank",
    "generated_at": datetime.datetime.now().isoformat(),
    "eval_k": EVAL_K,
    "metric_top_k": METRIC_TOP_K,
    "results_table": results_table,
    "per_query_results": per_query_results,
}

output_path = RESULTS_DIR / "04b_retrieval_sealion_rerank.json"
output_path.write_text(json.dumps(output, indent=2), encoding="utf-8")
output_path

WindowsPath('../data/processed/eval_results/04b_retrieval_sealion_rerank.json')